# Cassava navigation — compressed Phase A (GPU Colab)

**Runtime > Change runtime type > GPU**

Run every cell from top to bottom. This notebook is self-contained: upload only your dataset ZIP, or select it from Google Drive. No Git repository or local Mac files are required. It trains **one segmentation baseline only**. No classical ML, ANFIS, CNN navigation or transfer-learning navigation models are trained.

Classes are human-confirmed and fixed: **0 = path, 1 = cassava_leaves, 2 = ridge**.

Navigation targets are **provisional image-geometry research labels**, not recorded steering commands. Negative offset means image-left; positive means image-right. Continuous offset is the primary later regression/ANFIS target.

**Execution status:** notebook schema, Python syntax and helper safeguards are verified locally; GPU training and metrics are not claimed until you run this notebook. After execution, return the downloaded results ZIP and the executed `.ipynb` for review. Colab: **File > Download > Download .ipynb**. Save a copy to Drive before starting if you want notebook outputs retained across disconnections.

Helpers are embedded below and written to `src/`; their cells can be collapsed. They produce concise summaries; verbose training logs are saved in results. The source dataset is never deleted or edited.

## 1. Editable settings

In [ ]:
from pathlib import Path

SEED = 42
PROJECT = Path('/content/cassava-navigation-ai')
INPUT_METHOD = 'upload'          # 'upload' or 'drive'
ZIP_PATH = ''                    # Drive example: '/content/drive/MyDrive/cassava_dataset.zip'
ULTRALYTICS_VERSION = '8.4.138'   # Pinned release; checkpoint compatibility is tested at runtime.
IMAGE_SIZE = 640
EPOCHS = 60
BATCH_SIZE = 8
PATIENCE = 12
WORKERS = 2
LEARNING_RATE = 0.001
HORIZONTAL_FLIP = 0.0            # Enable (e.g. 0.5) only if mirroring is appropriate for your imagery.
LOWER_ROI_START = 0.65
PREDICTION_CONFIDENCE = 0.25      # Fixed before evaluation; no test-driven confidence tuning.
MAX_EXTRACT_GB = 10
RUN_NAME = 'baseline_seed42'
DOWNLOAD_RESULTS = True
# Training-only threshold policy. Bounds are provisional image-space assumptions, not robot settings.
THRESHOLD_POLICY = {
    'deadband_abs_offset_quantile': 0.35,
    'minimum_geometry_quantile': 0.10,
    'deadband_bounds': [0.05, 0.30],
    'area_bounds': [0.005, 0.10],
    'width_bounds': [0.05, 0.35],
    'continuity_bounds': [0.25, 0.90],
}
THRESHOLD_OVERRIDES = {}          # Optional named values: left_threshold, right_threshold,
                                # minimum_path_area_ratio, minimum_valid_path_width,
                                # minimum_path_continuity, stop_uncertain_threshold.
assert INPUT_METHOD in {'upload', 'drive'}
assert 0 < LOWER_ROI_START < 1 and 0 <= HORIZONTAL_FLIP <= 1
for folder in ['configs','data/extracted','data/processed','src/features','src/navigation',
               'models/segmentation_baseline','results/segmentation_baseline',
               'figures/segmentation_baseline','figures/navigation_targets','reports']:
    (PROJECT/folder).mkdir(parents=True,exist_ok=True)
print('Project:',PROJECT,'| Seed:',SEED,'| Epochs:',EPOCHS)


## 2. Install dependencies and detect GPU

The package release is pinned for reproducibility. The installed release is inspected before choosing its newest supported nano segmentation checkpoint. If a checkpoint cannot be loaded, compatibility attempts are recorded before trying an older nano model. Only the successfully selected architecture is trained. If GPU detection fails, change runtime type and rerun; this notebook never falls back to CPU training.

In [ ]:
import os, sys, subprocess, platform, json, random
os.environ['MPLCONFIGDIR'] = str(PROJECT/'.matplotlib-cache')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['WANDB_DISABLED'] = 'true'
subprocess.run([sys.executable,'-m','pip','install','--quiet',f'ultralytics=={ULTRALYTICS_VERSION}',
                'scikit-image==0.25.2','pandas','matplotlib','pyyaml','tabulate','nbformat'],check=True)
import torch, ultralytics, numpy as np, pandas as pd, yaml, cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); cv2.setRNGSeed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False
plt.rcParams.update({'savefig.dpi':220,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
ENVIRONMENT = {'python':platform.python_version(),'pytorch':torch.__version__,
               'cuda_available':torch.cuda.is_available(),'cuda_version':torch.version.cuda,
               'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               'ultralytics':ultralytics.__version__,'seed':SEED}
print(json.dumps(ENVIRONMENT,indent=2))
(PROJECT/'reports/environment.json').write_text(json.dumps(ENVIRONMENT,indent=2))
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(PROJECT/'reports/environment_freeze.txt').write_text(freeze)
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun.'
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT/'src'))


## 3. Embedded dataset validation helpers

In [ ]:
# Reusable helper: src/phase_a_io.py
helper_path = PROJECT / 'src/phase_a_io.py'
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text('"""Safe extraction, root discovery and read-only annotation validation."""\nfrom pathlib import Path, PurePosixPath\nimport hashlib\nimport json\nimport shutil\nimport stat\nimport zipfile\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport yaml\n\nNAMES = {0: \'path\', 1: \'cassava_leaves\', 2: \'ridge\'}\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.tif\', \'.tiff\', \'.webp\'}\n\ndef hidden(path: Path | PurePosixPath) -> bool:\n    """Exclude hidden/system paths without deleting their source files."""\n    return any(part.startswith((\'.\', \'__MACOSX\')) for part in path.parts)\n\ndef safe_extract(archive: Path, destination: Path, max_bytes: int) -> dict:\n    """Preflight every member; reject traversal, symlinks and oversized archives."""\n    destination.mkdir(parents=True, exist_ok=True)\n    digest = hashlib.sha256(archive.read_bytes()).hexdigest()\n    marker = destination / \'.phase_a_archive.json\'\n    if marker.exists():\n        old = json.loads(marker.read_text())\n        if old[\'sha256\'] != digest:\n            raise ValueError(\'Extraction directory belongs to another ZIP. Use a fresh runtime to avoid mixing datasets.\')\n        return old\n    if any(destination.iterdir()):\n        raise ValueError(\'Extraction is incomplete or directory is nonempty. Use a fresh runtime; no source files will be deleted.\')\n    ignored, extracted, seen = 0, 0, set()\n    with zipfile.ZipFile(archive) as z:\n        infos = z.infolist()\n        if sum(i.file_size for i in infos) > max_bytes:\n            raise ValueError(\'Archive exceeds MAX_EXTRACT_GB. Review archive before increasing the limit.\')\n        for info in infos:\n            name = PurePosixPath(info.filename)\n            if name.is_absolute() or \'..\' in name.parts or \'\\\\\' in info.filename or \':\' in info.filename:\n                raise ValueError(f\'Unsafe ZIP path: {info.filename!r}\')\n            if stat.S_ISLNK(info.external_attr >> 16):\n                raise ValueError(f\'ZIP symlink rejected: {info.filename!r}\')\n            if not info.is_dir():\n                if str(name).casefold() in seen:\n                    raise ValueError(f\'Duplicate ZIP destination: {name}\')\n                seen.add(str(name).casefold())\n            if not (destination / str(name)).resolve().is_relative_to(destination.resolve()):\n                raise ValueError(\'ZIP path escapes extraction root\')\n        for info in infos:\n            name = PurePosixPath(info.filename)\n            if hidden(name):\n                ignored += 1\n                continue\n            dest = destination / str(name)\n            if info.is_dir():\n                dest.mkdir(parents=True, exist_ok=True)\n            else:\n                dest.parent.mkdir(parents=True, exist_ok=True)\n                with z.open(info) as source, dest.open(\'wb\') as target:\n                    shutil.copyfileobj(source, target)\n                extracted += 1\n    result = dict(sha256=digest, ignored_archive_members=ignored, extracted_files=extracted)\n    marker.write_text(json.dumps(result, indent=2))\n    return result\n\ndef discover_root(extracted: Path) -> Path:\n    """Find one source YOLO root while ignoring generated project outputs.\n\n    A user may upload a full project ZIP containing both ``data/extracted/data``\n    and a derived ``data/processed/segmentation_dataset``. Generated roots are\n    excluded when exactly one source candidate remains. Ambiguous source roots\n    still stop execution instead of being selected heuristically.\n    """\n    roots = []\n    generated_parts = {\'processed\', \'segmentation_dataset\', \'models\', \'results\', \'figures\', \'runs\'}\n    for images in extracted.rglob(\'images\'):\n        base = images.parent\n        if hidden(images.relative_to(extracted)) or not images.is_dir():\n            continue\n        if all((images/s).is_dir() and (base/\'labels\'/s).is_dir() for s in [\'train\',\'test\']):\n            candidates = [s for s in [\'valid\',\'val\'] if (images/s).is_dir() and (base/\'labels\'/s).is_dir()]\n            if len(candidates) == 1:\n                roots.append(base)\n            elif len(candidates) > 1:\n                raise ValueError(f\'Both val and valid exist at {base}; resolve the ambiguous split explicitly.\')\n    roots = sorted(set(roots))\n    source_roots = [root for root in roots if not generated_parts.intersection(\n        part.casefold() for part in root.relative_to(extracted).parts)]\n    ignored = [root for root in roots if root not in source_roots]\n    if len(source_roots) == 1:\n        if ignored:\n            print(\'Ignored generated YOLO root(s):\', [str(p) for p in ignored])\n        return source_roots[0]\n    if not source_roots and len(roots) == 1:\n        print(\'Only a generated-style YOLO root was found; using it as the sole candidate.\')\n        return roots[0]\n    raise ValueError(\n        f\'Expected exactly one source YOLO root; found {len(source_roots)} source candidate(s): \'\n        f\'{source_roots}. Generated candidates ignored: {ignored}\'\n    )\n\ndef validate_dataset(root: Path, project: Path, archive_info: dict) -> pd.DataFrame:\n    """Keep size warnings; skip technical row errors in a separate derived view.\n\n    Missing/corrupt images are excluded with an audit record. Empty labels are\n    retained as explicit annotated-negative/uncertain samples. All-invalid labels\n    are excluded, never silently converted to negative training examples.\n    """\n    reports = project/\'reports\'; reports.mkdir(parents=True, exist_ok=True)\n    processed = project/\'data/processed\'; processed.mkdir(parents=True, exist_ok=True)\n    stage = processed/\'segmentation_dataset\'\n    rows, records, stats = [], [], []\n    valid_name = \'valid\' if (root/\'images/valid\').is_dir() else \'val\'\n    def issue(split: str, path: Path, kind: str, severity: str, line: int = 0, detail: str = \'\') -> None:\n        records.append(dict(split=split, path=str(path), line=line, issue=kind, severity=severity, detail=detail))\n    for original_split, split in [(\'train\',\'train\'),(valid_name,\'valid\'),(\'test\',\'test\')]:\n        idir, ldir = root/\'images\'/original_split, root/\'labels\'/original_split\n        all_images = sorted(p for p in idir.rglob(\'*\') if p.is_file() and p.suffix.lower() in EXTENSIONS)\n        images = [p for p in all_images if not hidden(p.relative_to(idir))]\n        labels = [p for p in ldir.rglob(\'*.txt\') if not hidden(p.relative_to(ldir))]\n        usable = 0\n        for path in all_images:\n            if hidden(path.relative_to(idir)):\n                issue(split,path,\'hidden_or_checkpoint\',\'excluded\')\n        seen_stems = set()\n        for path in images:\n            rel = path.relative_to(idir); stem_key = str(rel.with_suffix(\'\')).casefold()\n            if stem_key in seen_stems:\n                issue(split,path,\'duplicate_image_id\',\'error\')\n                continue\n            seen_stems.add(stem_key)\n            label = ldir/rel.with_suffix(\'.txt\')\n            if not label.exists():\n                issue(split,path,\'missing_label\',\'excluded\'); continue\n            try:\n                with Image.open(path) as im:\n                    im.verify()\n                with Image.open(path) as im:\n                    im.convert(\'RGB\').load()\n            except Exception as exc:\n                issue(split,path,\'corrupt_image\',\'excluded\',detail=str(exc)); continue\n            lines = label.read_text(encoding=\'utf-8\').splitlines()\n            accepted, invalid, warnings = [], 0, []\n            for line_no, line in enumerate(lines,1):\n                if not line.strip():\n                    continue\n                try:\n                    v=np.array([float(x) for x in line.split()])\n                    if len(v)<7 or (len(v)-1)%2 or not np.isfinite(v).all():\n                        raise ValueError(\'malformed_polygon\')\n                    if v[0] not in NAMES:\n                        raise ValueError(\'invalid_class_id\')\n                    if (v[1:]<0).any() or (v[1:]>1).any():\n                        raise ValueError(\'coordinates_outside_0_1\')\n                    points=v[1:].reshape(-1,2)\n                    area=abs(np.dot(points[:,0],np.roll(points[:,1],1))-np.dot(points[:,1],np.roll(points[:,0],1)))/2\n                    if len(np.unique(points,axis=0))<3 or area<=0:\n                        raise ValueError(\'degenerate_polygon\')\n                except ValueError as exc:\n                    invalid+=1; issue(split,label,str(exc),\'invalid_row_excluded\',line_no); continue\n                accepted.append(line.strip())\n                flag=\'extremely_small_polygon\' if area<1e-5 else \'extremely_large_polygon\' if area>.35 else None\n                if flag:\n                    warnings.append(flag);issue(split,label,flag,\'warning_retained\',line_no,detail=f\'normalized_area={area:.9g}\')\n            if invalid and not accepted:\n                issue(split,label,\'all_polygons_invalid\',\'excluded\');continue\n            if not accepted:\n                warnings.append(\'empty_label\');issue(split,label,\'empty_label\',\'warning_retained\')\n            if invalid:\n                warnings.append(\'technical_rows_excluded\')\n            dest_image=stage/\'images\'/split/rel;dest_label=stage/\'labels\'/split/rel.with_suffix(\'.txt\')\n            dest_image.parent.mkdir(parents=True,exist_ok=True);dest_label.parent.mkdir(parents=True,exist_ok=True)\n            if not dest_image.exists():shutil.copy2(path,dest_image)\n            dest_label.write_text(\'\\n\'.join(accepted)+ (\'\\n\' if accepted else \'\'))\n            rows.append(dict(image_id=split+\'/\'+rel.with_suffix(\'\').as_posix(),split=split,original_split=original_split,\n                             image_path=str(path.relative_to(project)),original_image_path=str(path.relative_to(project)),\n                             label_path=str(dest_label.relative_to(project)),training_image_path=str(dest_image.relative_to(project)),\n                             sha256=hashlib.sha256(path.read_bytes()).hexdigest(),warning_flags=\';\'.join(sorted(set(warnings))),extraction_status=\'usable\'))\n            usable+=1\n        image_stems={str(p.relative_to(idir).with_suffix(\'\')) for p in images}\n        for label in labels:\n            if str(label.relative_to(ldir).with_suffix(\'\')) not in image_stems:\n                issue(split,label,\'orphan_label\',\'warning\')\n        stats.append(dict(split=split,original_split=original_split,images=len(images),labels=len(labels),usable_images=usable,hidden_images=len(all_images)-len(images)))\n    mf=pd.DataFrame(rows)\n    if not mf.empty:\n        for digest, group in mf.groupby(\'sha256\'):\n            if len(group)>1:\n                kind=\'cross_split_duplicate_hash\' if group.split.nunique()>1 else \'within_split_duplicate_hash\'\n                for row in group.itertuples():issue(row.split,Path(row.original_image_path),kind,\'error\' if kind.startswith(\'cross\') else \'warning\')\n    records.append(dict(split=\'all\',path=\'ZIP\',line=0,issue=\'ignored_archive_members\',severity=\'info\',detail=str(archive_info[\'ignored_archive_members\'])))\n    quality=pd.DataFrame(records,columns=[\'split\',\'path\',\'line\',\'issue\',\'severity\',\'detail\'])\n    quality.to_csv(reports/\'phaseA_pretraining_quality_check.csv\',index=False)\n    pd.DataFrame(stats).to_csv(reports/\'dataset_split_summary.csv\',index=False)\n    mf.to_csv(processed/\'image_manifest.csv\',index=False)\n    display_summary=pd.DataFrame(stats)\n    print(display_summary.to_string(index=False))\n    print(\'Validation findings:\',quality.groupby([\'severity\',\'issue\']).size().to_dict())\n    print(\'Size warnings are retained; reference counts were 198 small and 18 large. Actual counts above are recomputed.\')\n    print(\'Nominal historical 603 training images included a checkpoint duplicate; 602 usable train images is expected after filtering. No reassignment.\')\n    if mf.empty or set(mf.split)!= {\'train\',\'valid\',\'test\'}:\n        raise ValueError(\'Every split must contain usable images; review the quality CSV.\')\n    if (quality.severity==\'error\').any():\n        raise ValueError(\'Duplicate IDs or cross-split image duplicates found. Resolve leakage before training; no automatic split changes.\')\n    dataset=dict(path=str(stage),train=str(stage/\'train.txt\'),val=str(stage/\'valid.txt\'),test=str(stage/\'test.txt\'),names=NAMES)\n    for split in [\'train\',\'valid\',\'test\']:\n        (stage/f\'{split}.txt\').write_text(\'\\n\'.join(str(project/p) for p in mf.loc[mf.split==split,\'training_image_path\'])+\'\\n\')\n    configs=project/\'configs\';configs.mkdir(exist_ok=True)\n    (configs/\'cassava_segmentation.yaml\').write_text(yaml.safe_dump(dataset,sort_keys=False))\n    (configs/\'classes.yaml\').write_text(yaml.safe_dump(dict(names=NAMES,status=\'confirmed\'),sort_keys=False))\n    return mf\n', encoding='utf-8')
print('Created', helper_path.relative_to(PROJECT))


## 4. Select and safely extract the dataset ZIP

**Option A:** leave `INPUT_METHOD = 'upload'`, run this cell, and select one ZIP. Its filename is detected automatically.

**Option B:** set `INPUT_METHOD = 'drive'` and edit `ZIP_PATH` near the top. Drive is mounted at `/content/drive`.

The archive is preflighted for unsafe paths, symlinks, duplicate destinations and size before extraction. Hidden/checkpoint files are ignored and counted. Existing extraction is reused only for the identical archive. For a different ZIP, use a fresh runtime.

In [ ]:
from phase_a_io import safe_extract, discover_root, validate_dataset
from google.colab import files, drive
if INPUT_METHOD == 'drive':
    drive.mount('/content/drive')
    archive = Path(ZIP_PATH).expanduser()
    assert archive.is_file() and archive.suffix.lower()=='.zip', 'Set ZIP_PATH to an existing Drive ZIP.'
else:
    if 'archive' not in globals() or not archive.is_file():
        uploaded = files.upload()
        zip_names = [name for name in uploaded if Path(name).suffix.lower()=='.zip']
        assert len(zip_names)==1, 'Upload exactly one dataset ZIP.'
        archive = Path(zip_names[0]).resolve()
        del uploaded
    print('Using uploaded ZIP:',archive.name)
ARCHIVE_INFO=safe_extract(archive,PROJECT/'data/extracted',int(MAX_EXTRACT_GB*1024**3))
DATASET_ROOT=discover_root(PROJECT/'data/extracted')
print('Detected YOLO root:',DATASET_ROOT)
print('Extraction:',ARCHIVE_INFO)


## 5. Pre-training quality check and confirmed classes

Validate readable images, labels, classes, polygon structure and coordinate ranges. Technical row errors are logged and excluded only from a **derived training copy**; images with wholly invalid annotations are excluded instead of being mislabeled as background. Empty labels are retained and flagged for review. Cross-split exact duplicates and duplicate IDs stop training.

The previous **198 very small and 18 very large polygons are warnings, not automatic errors**; this run recomputes and reports their actual counts. Historical 603 train images included one checkpoint duplicate, so 602 usable train images is expected after artifact filtering; validation/test membership is retained. Exact hash checks cannot rule out nearby video frames or same-field leakage.

In [ ]:
MANIFEST=validate_dataset(DATASET_ROOT,PROJECT,ARCHIVE_INFO)
assert len(MANIFEST[MANIFEST.split=='test'])>=20, 'At least 20 usable test images are required for requested diagnostics.'
assert len(MANIFEST)>=30, 'At least 30 usable images are required for navigation overlays.'
print('Confirmed classes:',yaml.safe_load((PROJECT/'configs/classes.yaml').read_text()))
print('Saved reports/phaseA_pretraining_quality_check.csv and validated dataset YAML.')


## 6. Embedded reusable feature, target, segmentation and analysis code

In [ ]:
# Reusable helper: src/features/feature_extraction.py
helper_path = PROJECT / 'src/features/feature_extraction.py'
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text('"""Shared native-resolution geometry and image descriptors for GT/predicted polygons.\nCoordinates use pixel centres; x increases right, y down. Empty geometry is NaN.\nUnion areas avoid double-counting overlapping instances. Original data is read-only.\n"""\nfrom pathlib import Path\nimport hashlib\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom skimage.feature import graycomatrix, graycoprops, local_binary_pattern\nfrom skimage.measure import shannon_entropy\n\nROOT = Path(__file__).resolve().parents[2]\nNAMES = [\'path\', \'cassava_leaves\', \'ridge\']\n\ndef config() -> dict:\n    return yaml.safe_load((ROOT/\'configs/features.yaml\').read_text())\n\ndef manifest() -> pd.DataFrame:\n    """Read validated canonical IDs and source paths produced before training."""\n    return pd.read_csv(ROOT/\'data/processed/image_manifest.csv\',keep_default_na=False)\n\n\ndef polygons(label: Path) -> dict:\n    result = {i:[] for i in range(3)}\n    for line in Path(label).read_text().splitlines():\n        v = np.array([float(x) for x in line.split()])\n        if len(v)<7 or (len(v)-1)%2 or not np.isfinite(v).all() or v[0] not in result or (v[1:]<0).any() or (v[1:]>1).any():\n            raise ValueError(f\'Technically invalid polygon in {label}\')\n        p = v[1:].reshape(-1,2)\n        if cv2.contourArea(p.astype(np.float32)) <= 0:\n            raise ValueError(f\'Zero-area polygon in {label}\')\n        result[int(v[0])].append(p)\n    return result\n\ndef masks_from_polygons(polys: dict, h: int, w: int) -> list:\n    masks = []\n    for c in range(3):\n        m = np.zeros((h,w),np.uint8)\n        for p in polys[c]:\n            q = np.rint(np.asarray(p)*[w-1,h-1]).astype(np.int32)\n            # Fill independently: OpenCV\'s collective even/odd fill can erase overlaps.\n            cv2.fillPoly(m,[q],1)\n        masks.append(m)\n    return masks\n\ndef extract(image: np.ndarray, polys: dict) -> dict:\n    cfg=config(); h,w=image.shape[:2]; masks=masks_from_polygons(polys,h,w); f={}\n    def put(name,value): f[name]=float(value)\n    for c,name in enumerate(NAMES):\n        m=masks[c]; yy,xx=np.nonzero(m); points=[np.asarray(p)*[w-1,h-1] for p in polys[c]]\n        areas=[cv2.contourArea(p.astype(np.float32)) for p in points]\n        values=dict(polygon_count=len(points),mask_area=m.sum(),area_ratio=m.mean(),largest_polygon_area=max(areas) if areas else np.nan,mean_polygon_area=np.mean(areas) if areas else np.nan,centroid_x=xx.mean() if len(xx) else np.nan,centroid_y=yy.mean() if len(yy) else np.nan,bounding_width=np.ptp(xx)+1 if len(xx) else np.nan,bounding_height=np.ptp(yy)+1 if len(yy) else np.nan,perimeter=sum(cv2.arcLength(p.astype(np.float32),True) for p in points) if points else np.nan,leftmost=xx.min() if len(xx) else np.nan,rightmost=xx.max() if len(xx) else np.nan,topmost=yy.min() if len(yy) else np.nan,bottommost=yy.max() if len(yy) else np.nan)\n        values[\'aspect_ratio\']=values[\'bounding_width\']/values[\'bounding_height\']\n        for k,v in values.items(): put(f\'{name}_{k}\',v)\n    path,leaf,ridge=masks; start=int(h*cfg[\'lower_roi_start\']); roi=path[start:]\n    left=[];right=[]; widths=[]\n    for row in roi:\n        x=np.flatnonzero(row)\n        if len(x): left.append(x.min());right.append(x.max());widths.append(x.max()-x.min()+1)\n    put(\'path_left_boundary\',np.median(left) if left else np.nan)\n    put(\'path_right_boundary\',np.median(right) if right else np.nan)\n    put(\'path_center_x\',(f[\'path_left_boundary\']+f[\'path_right_boundary\'])/2)\n    put(\'path_width_lower\',np.median(widths)/w if widths else np.nan)\n    put(\'path_width\',f[\'path_bounding_width\']/w)\n    put(\'normalized_path_offset\',(f[\'path_center_x\']-w/2)/(w/2))\n    put(\'absolute_path_deviation\',abs(f[\'normalized_path_offset\']))\n    put(\'path_continuity\',np.any(roi,axis=1).mean())\n    put(\'missing_path\',not path.any())\n    for y in cfg[\'width_rows\']:\n        x=np.flatnonzero(path[min(h-1,int(y*h))]);put(f\'path_width_y{y}\',(x.max()-x.min()+1)/w if len(x) else np.nan)\n    for name,m in [(\'cassava\',leaf),(\'ridge\',ridge)]:\n        put(name+\'_coverage\',m.mean());put(name+\'_left_coverage\',m[:,:w//2].mean());put(name+\'_right_coverage\',m[:,w//2:].mean())\n        put(name+\'_imbalance\',f[name+\'_left_coverage\']-f[name+\'_right_coverage\'])\n    put(\'cassava_lower_obstruction\',leaf[start:].mean());put(\'cassava_density\',len(polys[1])/(h*w/1e6))\n    y,x=np.nonzero(ridge)\n    orientation=np.nan\n    if len(x)>2:\n        vals,vecs=np.linalg.eigh(np.cov(np.vstack([x,y])))\n        if vals[-1]>1.05*vals[0]: orientation=np.degrees(np.arctan2(vecs[1,-1],vecs[0,-1]))%180\n    put(\'ridge_orientation\',orientation)\n    put(\'ridge_proximity_to_path\',cv2.distanceTransform(1-path,cv2.DIST_L2,5)[ridge.astype(bool)].mean()/np.hypot(w,h) if path.any() and ridge.any() else np.nan)\n    put(\'ridge_path_overlap\',(ridge & path).sum()/ridge.sum() if ridge.any() else np.nan)\n    put(\'ridge_path_centroid_distance\',np.hypot(f[\'ridge_centroid_x\']-f[\'path_centroid_x\'],f[\'ridge_centroid_y\']-f[\'path_centroid_y\'])/np.hypot(w,h))\n    rgb=image.astype(float)/255; hsv=cv2.cvtColor(image,cv2.COLOR_RGB2HSV).astype(float)/[179,255,255]\n    for space,a in [(\'rgb\',rgb),(\'hsv\',hsv)]:\n        for j in range(3):\n            put(f\'{space}_{j}_mean\',a[:,:,j].mean());put(f\'{space}_{j}_std\',a[:,:,j].std())\n            hist=np.histogram(a[:,:,j],bins=cfg[\'histogram_bins\'],range=(0,1))[0]/(h*w)\n            for b,v in enumerate(hist):put(f\'{space}_{j}_hist_{b}\',v)\n    r,g,b=rgb.transpose(2,0,1)\n    put(\'excess_green\',(2*g-r-b).mean());put(\'excess_red\',(1.4*r-g).mean());put(\'normalized_green_red\',np.mean((g-r)/(g+r+1e-8)))\n    gray=cv2.cvtColor(image,cv2.COLOR_RGB2GRAY); small=cv2.resize(gray,(cfg[\'texture_size\'],)*2,interpolation=cv2.INTER_AREA)\n    quant=(small.astype(float)*cfg[\'glcm_levels\']/256).astype(np.uint8)\n    glcm=graycomatrix(quant,[1],[0,np.pi/4,np.pi/2,3*np.pi/4],levels=cfg[\'glcm_levels\'],symmetric=True,normed=True)\n    for prop in [\'contrast\',\'correlation\',\'energy\',\'homogeneity\']:put(\'glcm_\'+prop,graycoprops(glcm,prop).mean())\n    put(\'texture_entropy\',shannon_entropy(small))\n    lbp=local_binary_pattern(small,8,1,method=\'uniform\')\n    for i,v in enumerate(np.histogram(lbp,bins=np.arange(11))[0]/lbp.size):put(f\'lbp_hist_{i}\',v)\n    put(\'lbp_mean\',lbp.mean());put(\'lbp_std\',lbp.std())\n    edges=cv2.Canny(gray,*cfg[\'canny_thresholds\']);put(\'edge_density\',(edges>0).mean())\n    contours,_=cv2.findContours(edges,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)\n    put(\'edge_contour_count\',len(contours));put(\'edge_contour_mean_area\',np.mean([cv2.contourArea(c) for c in contours]) if contours else np.nan)\n    put(\'edge_contour_mean_perimeter\',np.mean([cv2.arcLength(c,True) for c in contours]) if contours else np.nan)\n    lines=cv2.HoughLinesP(edges,1,np.pi/180,threshold=40,minLineLength=30,maxLineGap=10)\n    angle=np.nan\n    if lines is not None:\n        lines=lines.reshape(-1,4).astype(float);d=lines[:,2:]-lines[:,:2];angles=np.arctan2(d[:,1],d[:,0])%np.pi\n        hist,bins=np.histogram(angles,bins=18,range=(0,np.pi),weights=np.linalg.norm(d,axis=1));angle=np.degrees((bins[hist.argmax()]+bins[hist.argmax()+1])/2)\n    put(\'edge_dominant_orientation\',angle)\n    f.update({k+\'_missing\':int(not np.isfinite(v)) for k,v in list(f.items())})\n    return f\n\ndef run(predicted: dict | None = None) -> pd.DataFrame:\n    rows=[]; mf=manifest(); (ROOT/\'data/processed\').mkdir(exist_ok=True)\n    for i,row in enumerate(mf.to_dict(\'records\')):\n        image=cv2.cvtColor(cv2.imread(str(ROOT/row[\'image_path\'])),cv2.COLOR_BGR2RGB)\n        ps=polygons(ROOT/row[\'label_path\']) if predicted is None else predicted[row[\'image_id\']]\n        values=extract(image,ps)\n        flags=[row.get(\'warning_flags\',\'\')]\n        if values[\'missing_path\']: flags.append(\'missing_path\')\n        if values[\'path_center_x_missing\']: flags.append(\'missing_lower_roi_center\')\n        rows.append(dict(image_id=row[\'image_id\'],original_image_path=row[\'original_image_path\'],split=row[\'split\'],extraction_status=\'ok\',warning_flags=\';\'.join(x for x in flags if x),image_width=image.shape[1],image_height=image.shape[0],**values))\n        if i%100==0: print(f\'Features: {i}/{len(mf)}\',flush=True)\n    df=pd.DataFrame(rows); name=\'features_ground_truth.csv\' if predicted is None else \'features_predicted_masks.csv\'\n    df.to_csv(ROOT/\'data/processed\'/name,index=False)\n    return df\n\nif __name__==\'__main__\': run()\n', encoding='utf-8')
print('Created', helper_path.relative_to(PROJECT))


In [ ]:
# Reusable helper: src/navigation/target_generation.py
helper_path = PROJECT / 'src/navigation/target_generation.py'
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text('"""Provisional image-space research labels; these are not robot commands."""\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport yaml\nROOT=Path(__file__).resolve().parents[2]\n\ndef generate(features: pd.DataFrame, cfg: dict) -> pd.DataFrame:\n    if cfg[\'smoothing\'][\'enabled\']:\n        raise ValueError(\'Temporal smoothing requires validated sequence IDs; unavailable here.\')\n    rows=[]\n    for f in features.to_dict(\'records\'):\n        flags=[]\n        annotation_flags=f.get(\'warning_flags\',\'\')\n        if isinstance(annotation_flags,str) and \'technical_rows_excluded\' in annotation_flags:\n            flags.append(\'technical_rows_excluded\')\n        for key,limit,flag in [(\'path_area_ratio\',cfg[\'minimum_path_area_ratio\'],\'low_area\'),(\'path_width_lower\',cfg[\'minimum_valid_path_width\'],\'narrow_path\'),(\'path_continuity\',cfg[\'minimum_path_continuity\'],\'discontinuous_path\')]:\n            if not np.isfinite(f[key]) or f[key]<limit: flags.append(flag)\n        offset=f[\'normalized_path_offset\']\n        if not np.isfinite(offset): flags.append(\'missing_path_center\')\n        quality=min(np.clip(f[\'path_area_ratio\']/cfg[\'minimum_path_area_ratio\'],0,1),np.clip(f[\'path_width_lower\']/cfg[\'minimum_valid_path_width\'],0,1) if np.isfinite(f[\'path_width_lower\']) else 0,f[\'path_continuity\'])\n        if quality<cfg[\'stop_uncertain_threshold\']: flags.append(\'low_quality\')\n        label=\'stop_or_uncertain\' if flags else (\'left\' if offset<cfg[\'left_threshold\'] else \'right\' if offset>cfg[\'right_threshold\'] else \'forward\')\n        rows.append(dict(image_id=f[\'image_id\'],split=f[\'split\'],path_center_x=f[\'path_center_x\'],image_center_x=f[\'image_width\']/2,normalized_offset=offset,absolute_offset=abs(offset),path_width=f[\'path_width_lower\'],path_area_ratio=f[\'path_area_ratio\'],path_continuity=f[\'path_continuity\'],discrete_target=label,continuous_target=offset,target_confidence=quality,target_status=\'provisional_geometry_derived\',warning_flags=\';\'.join(flags),continuous_target_missing=int(not np.isfinite(offset))))\n    return pd.DataFrame(rows)\n\ndef run() -> pd.DataFrame:\n    f=pd.read_csv(ROOT/\'data/processed/features_ground_truth.csv\');cfg=yaml.safe_load((ROOT/\'configs/navigation_targets.yaml\').read_text())\n    feature_cfg=yaml.safe_load((ROOT/\'configs/features.yaml\').read_text())\n    if cfg[\'lower_roi_start\']!=feature_cfg[\'lower_roi_start\']: raise ValueError(\'ROI mismatch: regenerate features with the target ROI\')\n    t=generate(f,cfg);t.to_csv(ROOT/\'data/processed/navigation_targets.csv\',index=False);return t\nif __name__==\'__main__\':run()\n', encoding='utf-8')
print('Created', helper_path.relative_to(PROJECT))


In [ ]:
# Reusable helper: src/phase_a_segmentation.py
helper_path = PROJECT / 'src/phase_a_segmentation.py'
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text('"""Train one compatible nano segmentation model and export actual evaluations."""\nfrom pathlib import Path\nimport contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport shutil\nimport sys\nimport time\nimport cv2\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport yaml\n\n\ndef number(value: object) -> float:\n    """Return finite scalar or NaN; never manufacture an unavailable metric."""\n    try:\n        value=float(value)\n        return value if math.isfinite(value) else np.nan\n    except (TypeError,ValueError):return np.nan\n\n\ndef select_model(project: Path) -> tuple:\n    """Test architecture presence and successful checkpoint loading before training."""\n    import ultralytics\n    from ultralytics import YOLO\n    package=Path(ultralytics.__file__).parent\n    candidates=[(\'yolo26n-seg.pt\',\'26/yolo26-seg.yaml\'),(\'yolo11n-seg.pt\',\'11/yolo11-seg.yaml\'),(\'yolov8n-seg.pt\',\'v8/yolov8-seg.yaml\')]\n    attempts=[]\n    for checkpoint,architecture in candidates:\n        if not (package/\'cfg/models\'/architecture).exists():\n            attempts.append(dict(model=checkpoint,status=\'architecture_not_in_installed_version\'));continue\n        try:\n            model=YOLO(checkpoint)\n            if model.task!=\'segment\':raise ValueError(\'Checkpoint is not a segmentation model\')\n            attempts.append(dict(model=checkpoint,status=\'checkpoint_loaded\'))\n            pd.DataFrame(attempts).to_csv(project/\'reports/model_compatibility_check.csv\',index=False)\n            return model,checkpoint\n        except Exception as exc:\n            attempts.append(dict(model=checkpoint,status=\'load_failed\',reason=str(exc)[:500]))\n    pd.DataFrame(attempts).to_csv(project/\'reports/model_compatibility_check.csv\',index=False)\n    raise RuntimeError(\'No compatible nano segmentation checkpoint loaded. Check network/package errors in the compatibility report; no model trained.\')\n\n\ndef train_one(project: Path, config: dict, manifest: pd.DataFrame, run_name: str) -> tuple:\n    """Reuse only a completed run with matching data/config; otherwise train once."""\n    import torch\n    import ultralytics\n    from ultralytics import YOLO\n    if not torch.cuda.is_available():raise RuntimeError(\'GPU required: Runtime > Change runtime type > GPU\')\n    out=project/\'results/segmentation_baseline\';models=project/\'models/segmentation_baseline\'\n    out.mkdir(parents=True,exist_ok=True);models.mkdir(parents=True,exist_ok=True)\n    # Include raw label content hashes: changed annotations invalidate a cached training run.\n    labels=[hashlib.sha256((project/p).read_bytes()).hexdigest() for p in manifest.label_path]\n    payload=dict(config=config,images=manifest.sha256.tolist(),labels=labels,version=ultralytics.__version__,run_name=run_name)\n    signature=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()\n    complete=out/\'training_complete.json\'\n    if complete.exists():\n        state=json.loads(complete.read_text())\n        if state[\'signature\']!=signature:raise ValueError(\'Completed run has different data/config. Use a fresh runtime for a new experiment; no overwrite.\')\n        for key in [\'best.pt\',\'last.pt\']:\n            if not (models/key).exists():raise FileNotFoundError(f\'Missing completed checkpoint: {key}\')\n        print(\'Reusing completed segmentation training with matching configuration and dataset.\')\n        return YOLO(models/\'best.pt\'),state\n    run_dir=out/run_name\n    if run_dir.exists():raise RuntimeError(\'Incomplete training run exists. Preserve last.pt and restart in a fresh runtime, or resume explicitly outside this notebook. Do not mix partial runs.\')\n    # Disable environment-dependent implicit Albumentations; explicit HSV/geometry settings below remain active.\n    import ultralytics.data.augment as augment\n    def disabled_albumentations(self, *args, **kwargs):\n        self.p=0.0;self.transform=None;self.contains_spatial=False\n    augment.Albumentations.__init__=disabled_albumentations\n    model,name=select_model(project)\n    resolved={**config,\'model\':name,\'device\':0,\'ultralytics_version\':ultralytics.__version__,\'implicit_albumentations\':False}\n    (project/\'configs/yolo_baseline.yaml\').write_text(yaml.safe_dump(resolved,sort_keys=False))\n    best_epochs=[]\n    console=sys.stdout\n    def record_best(trainer):\n        if trainer.fitness==trainer.best_fitness:best_epochs.append(int(trainer.epoch)+1)\n    def progress(trainer):\n        epoch=int(trainer.epoch)+1\n        if epoch==1 or epoch%5==0:\n            print(f\'Epoch {epoch}/{config["epochs"]}; validation fitness={number(trainer.fitness):.4f}\',file=console,flush=True)\n    model.add_callback(\'on_model_save\',record_best)\n    model.add_callback(\'on_fit_epoch_end\',progress)\n    log=out/\'training.log\'\n    try:\n        with log.open(\'w\') as handle,contextlib.redirect_stdout(handle),contextlib.redirect_stderr(handle):\n            model.train(data=str(project/\'configs/cassava_segmentation.yaml\'),project=str(out),name=run_name,exist_ok=False,device=0,plots=True,**config)\n    except Exception:\n        print(\'Training stopped. Last log lines:\\n\'+\'\\n\'.join(log.read_text(errors=\'replace\').splitlines()[-15:]))\n        raise\n    for source,name_out in [(model.trainer.best,\'best.pt\'),(model.trainer.last,\'last.pt\')]:\n        source=Path(source)\n        if not source.exists():raise FileNotFoundError(source)\n        shutil.copy2(source,models/name_out)\n    state=dict(signature=signature,architecture=name,best_epoch=best_epochs[-1] if best_epochs else None,run_dir=str(model.trainer.save_dir),seed=config[\'seed\'])\n    complete.write_text(json.dumps(state,indent=2))\n    print(f\'Trained {name}; best checkpoint epoch: {state["best_epoch"]}. Best and last checkpoints saved.\')\n    return YOLO(models/\'best.pt\'),state\n\n\ndef metric_values(metrics: object, group: str) -> dict:\n    """Inspect both typed metric objects and results_dict across Ultralytics versions."""\n    obj=getattr(metrics,\'seg\' if group==\'mask\' else \'box\',None)\n    raw=getattr(metrics,\'results_dict\',{}) or {}\n    suffix=\'M\' if group==\'mask\' else \'B\'\n    names=[\'precision\',\'recall\',\'map50\',\'map50_95\']\n    keys=[\'precision\',\'recall\',\'mAP50\',\'mAP50-95\']\n    values=[np.nan]*4\n    if obj is not None and callable(getattr(obj,\'mean_results\',None)):\n        try:values=list(obj.mean_results())[:4]\n        except (AttributeError,TypeError,ValueError):pass\n    values=(values+[np.nan]*4)[:4]\n    result={}\n    for name,key,value in zip(names,keys,values):\n        val=number(value)\n        if not np.isfinite(val):val=number(raw.get(f\'metrics/{key}({suffix})\'))\n        result[f\'{group}_{name}\']=val\n    return result\n\n\ndef evaluate(project: Path, net: object, state: dict, config: dict) -> tuple:\n    """Evaluate frozen best checkpoint separately on validation and untouched test."""\n    out=project/\'results/segmentation_baseline\';figdir=project/\'figures/segmentation_baseline\'\n    figdir.mkdir(parents=True,exist_ok=True)\n    rows, per,availability=[],[],[]\n    names={0:\'path\',1:\'cassava_leaves\',2:\'ridge\'}\n    for split,api_split in [(\'valid\',\'val\'),(\'test\',\'test\')]:\n        name=f\'eval_{split}\'\n        with (out/f\'{name}.log\').open(\'w\') as log,contextlib.redirect_stdout(log),contextlib.redirect_stderr(log):\n            metrics=net.val(data=str(project/\'configs/cassava_segmentation.yaml\'),split=api_split,device=0,imgsz=config[\'imgsz\'],batch=config[\'batch\'],workers=config[\'workers\'],seed=config[\'seed\'],plots=True,project=str(out),name=name,exist_ok=True)\n        raw=getattr(metrics,\'results_dict\',{}) or {}\n        (out/f\'{split}_raw_metrics.json\').write_text(json.dumps({str(k):number(v) if np.isfinite(number(v)) else None for k,v in raw.items()},indent=2))\n        rows.append(dict(status=\'trained\',split=split,model=state[\'architecture\'],best_epoch=state[\'best_epoch\'],parameter_count=sum(p.numel() for p in net.model.parameters()),model_size_mb=(project/\'models/segmentation_baseline/best.pt\').stat().st_size/1e6,**metric_values(metrics,\'mask\'),**metric_values(metrics,\'box\')))\n        for c,label in names.items():\n            row=dict(split=split,class_id=c,class_name=label)\n            for group,attribute in [(\'mask\',\'seg\'),(\'box\',\'box\')]:\n                obj=getattr(metrics,attribute,None)\n                indices=list(getattr(obj,\'ap_class_index\',[])) if obj is not None else []\n                vals=[np.nan]*4\n                if c in indices and callable(getattr(obj,\'class_result\',None)):\n                    try:vals=list(obj.class_result(indices.index(c)))[:4]\n                    except (AttributeError,ValueError,TypeError,IndexError):pass\n                for key,value in zip([\'precision\',\'recall\',\'map50\',\'map50_95\'],(vals+[np.nan]*4)[:4]):row[f\'{group}_{key}\']=number(value)\n            per.append(row)\n        directory=out/name\n        pngs=list(directory.glob(\'*.png\'))\n        for path in pngs:shutil.copy2(path,figdir/f\'{split}_{path.name}\')\n        for label,pattern in [(\'precision_recall\',\'*PR_curve*\'),(\'f1_confidence\',\'*F1_curve*\'),(\'confusion_matrix\',\'*confusion_matrix*\')]:\n            available=bool(list(directory.glob(pattern)))\n            availability.append(dict(split=split,figure=label,status=\'generated\' if available else \'unavailable_from_installed_API\'))\n        print(f\'{split}: mask mAP50={rows[-1]["mask_map50"]:.4f}; mAP50–95={rows[-1]["mask_map50_95"]:.4f}\')\n    table=pd.DataFrame(rows);classes=pd.DataFrame(per)\n    table.to_csv(out/\'metrics.csv\',index=False);classes.to_csv(out/\'per_class_metrics.csv\',index=False)\n    pd.DataFrame(availability).to_csv(out/\'figure_availability.csv\',index=False)\n    history_path=Path(state[\'run_dir\'])/\'results.csv\'\n    if history_path.exists():\n        history=pd.read_csv(history_path);history.columns=history.columns.str.strip()\n        for split in [\'train\',\'val\']:\n            columns=[c for c in history if c.startswith(split+\'/\') and \'loss\' in c]\n            if columns:\n                fig,ax=plt.subplots(figsize=(8,4))\n                for c in columns:ax.plot(history[\'epoch\'],history[c],label=c)\n                ax.set(xlabel=\'Epoch\',ylabel=\'Loss\');ax.legend();fig.tight_layout();fig.savefig(figdir/f\'{split}_loss_curves.png\',dpi=240);plt.close(fig)\n    fig,ax=plt.subplots(figsize=(8,4));classes[classes.split==\'valid\'].plot.bar(x=\'class_name\',y=[\'mask_map50\',\'mask_map50_95\'],ax=ax,rot=0);ax.set(ylabel=\'Validation mask AP\',ylim=(0,1));fig.tight_layout();fig.savefig(figdir/\'per_class_performance.png\',dpi=240);plt.close(fig)\n    return table,classes\n\n\ndef predict_all(project: Path, net: object, manifest: pd.DataFrame, image_size: int, confidence: float) -> tuple:\n    """Save actual polygons, batch-one latency and all test prediction images."""\n    from features.feature_extraction import polygons,masks_from_polygons\n    out=project/\'results/segmentation_baseline\';figdir=project/\'figures/segmentation_baseline\'\n    testdir=figdir/\'test_predictions\';testdir.mkdir(exist_ok=True)\n    prediction_dir=out/\'predicted_polygons\';prediction_dir.mkdir(exist_ok=True)\n    polygons_by_id,records={},[]\n    source=str(project/manifest.iloc[0].image_path)\n    for _ in range(3):net.predict(source,imgsz=image_size,device=0,conf=confidence,verbose=False)\n    for i,row in enumerate(manifest.to_dict(\'records\')):\n        start=time.perf_counter()\n        result=net.predict(str(project/row[\'image_path\']),imgsz=image_size,device=0,conf=confidence,verbose=False,retina_masks=True)[0]\n        wall=(time.perf_counter()-start)*1000\n        ps={c:[] for c in range(3)}\n        if result.masks is not None and result.boxes is not None:\n            for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.masks.xyn):ps[int(c)].append(np.asarray(p))\n        polygons_by_id[row[\'image_id\']]=ps\n        file=prediction_dir/(row[\'image_id\']+\'.json\');file.parent.mkdir(parents=True,exist_ok=True)\n        file.write_text(json.dumps({c:[p.tolist() for p in points] for c,points in ps.items()}))\n        h,w=result.orig_shape\n        gt=masks_from_polygons(polygons(project/row[\'label_path\']),h,w);pred=masks_from_polygons(ps,h,w)\n        ious=[(a&b).sum()/(a|b).sum() for a,b in zip(gt,pred) if (a|b).any()]\n        speed=getattr(result,\'speed\',{}) or {}\n        vals=[number(speed.get(k)) for k in [\'preprocess\',\'inference\',\'postprocess\']]\n        latency=sum(vals) if all(np.isfinite(vals)) else np.nan\n        rec=dict(image_id=row[\'image_id\'],split=row[\'split\'],preprocess_ms=vals[0],model_inference_ms=vals[1],postprocess_ms=vals[2],inference_ms=latency,wall_ms=wall,mean_union_iou=np.mean(ious) if ious else np.nan,prediction_figure=\'\')\n        if row[\'split\']==\'test\':\n            image_file=testdir/(row[\'image_id\'].replace(\'/\',\'__\')+\'.png\')\n            cv2.imwrite(str(image_file),result.plot())\n            rec[\'prediction_figure\']=str(image_file.relative_to(project))\n        records.append(rec)\n        if (i+1)%100==0:print(f\'Predicted {i+1}/{len(manifest)} images\')\n    table=pd.DataFrame(records);table.to_csv(out/\'inference_results.csv\',index=False)\n    metrics=pd.read_csv(out/\'metrics.csv\')\n    for i,row in metrics.iterrows():\n        sample=table[table.split==row.split]\n        metrics.loc[i,\'inference_ms\']=sample.inference_ms.mean()\n        metrics.loc[i,\'fps\']=1000/sample.inference_ms.mean() if sample.inference_ms.mean()>0 else np.nan\n        metrics.loc[i,\'wall_ms\']=sample.wall_ms.mean()\n    metrics.to_csv(out/\'metrics.csv\',index=False)\n    test=table[table.split==\'test\'];failures=test.nsmallest(min(9,len(test)),\'mean_union_iou\').copy()\n    failures[\'reason\']=\'lowest semantic union IoU; relative diagnostic ranking, not a controller-failure label\'\n    failures.to_csv(out/\'failure_cases.csv\',index=False)\n    representative=[];used=set()\n    # Select each named class, mixed scenes, and difficult scenes without tuning on test.\n    for cls,label in [(0,\'path\'),(1,\'cassava_leaves\'),(2,\'ridge\'),(None,\'mixed scene\')]:\n        for row in test.sort_values(\'mean_union_iou\',ascending=False).itertuples():\n            ps=polygons_by_id[row.image_id]\n            eligible=bool(ps[cls]) if cls is not None else all(ps[c] for c in range(3))\n            if eligible and row.image_id not in used:\n                representative.append((row,label));used.add(row.image_id);break\n    for row in failures.head(2).itertuples():\n        representative.append((row,\'difficult: low union IoU\'));used.add(row.image_id)\n    if not representative:\n        representative=[(r,\'prediction (no class detected)\') for r in test.head(6).itertuples()]\n    def panel(items: list, path: Path, title: str) -> None:\n        columns=3;count=len(items);fig,axes=plt.subplots(max(1,int(np.ceil(count/columns))),columns,figsize=(15,5*max(1,int(np.ceil(count/columns)))),squeeze=False)\n        for ax in axes.flat:ax.axis(\'off\')\n        for ax,(row,label) in zip(axes.flat,items):\n            ax.imshow(plt.imread(project/row.prediction_figure));ax.set_title(f\'{label}\\nunion IoU={row.mean_union_iou:.3f}\',fontsize=11)\n        fig.suptitle(title,fontsize=16);fig.tight_layout();fig.savefig(path,dpi=220);plt.close(fig)\n    panel(representative,figdir/\'test_prediction_examples.png\',\'Actual test segmentation predictions\')\n    panel([(r,\'difficult test scene\') for r in failures.itertuples()],figdir/\'failure_cases.png\',\'Lowest semantic-union IoU examples\')\n    pd.DataFrame([dict(image_id=r.image_id,category=label) for r,label in representative]).to_csv(out/\'representative_prediction_manifest.csv\',index=False)\n    print(f\'Predictions complete: {len(table)} images; {len(test)} test examples. Absent predicted classes cannot have representative examples.\')\n    return polygons_by_id,table\n', encoding='utf-8')
print('Created', helper_path.relative_to(PROJECT))


In [ ]:
# Reusable helper: src/phase_a_analysis.py
helper_path = PROJECT / 'src/phase_a_analysis.py'
helper_path.parent.mkdir(parents=True, exist_ok=True)
helper_path.write_text('"""Reproducible quality analysis; selection statistics use training rows only."""\nimport os\nfrom pathlib import Path\nos.environ.setdefault(\'MPLCONFIGDIR\',str(Path(__file__).resolve().parents[1]/\'.matplotlib-cache\'))\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport yaml\nfrom features.feature_extraction import ROOT,manifest,polygons,masks_from_polygons\nfrom navigation.target_generation import run as targets\nplt.rcParams.update({\'figure.dpi\':130,\'savefig.dpi\':220,\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False})\n\ndef save(fig: object, path: Path) -> None:\n    fig.tight_layout();fig.savefig(path);plt.close(fig)\n\ndef run() -> dict:\n    roi_start=yaml.safe_load((ROOT/\'configs/features.yaml\').read_text())[\'lower_roi_start\']\n    f=pd.read_csv(ROOT/\'data/processed/features_ground_truth.csv\'); t=targets();mf=manifest()\n    out=ROOT/\'reports\';figdir=ROOT/\'figures/navigation_targets\'; qdir=ROOT/\'figures/feature_quality\';qdir.mkdir(exist_ok=True)\n    numeric=f.select_dtypes(include=\'number\').drop(columns=[\'image_width\',\'image_height\']);train=numeric[f.split==\'train\']\n    missing=pd.DataFrame({\'feature\':numeric.columns,\'missing_count\':numeric.isna().sum().values,\'missing_rate\':numeric.isna().mean().values})\n    missing.to_csv(out/\'feature_missing_values.csv\',index=False)\n    summary=pd.DataFrame({\'feature\':numeric.columns,\'train_unique\':train.nunique().values,\'train_dominant_fraction\':[train[c].value_counts(dropna=False,normalize=True).iloc[0] for c in train]})\n    summary[\'constant\']=summary.train_unique<=1;summary[\'near_constant\']=summary.train_dominant_fraction>=.99\n    summary.to_csv(out/\'feature_extraction_summary.csv\',index=False)\n    summary[summary.constant].to_csv(out/\'constant_features.csv\',index=False);summary[summary.near_constant & ~summary.constant].to_csv(out/\'near_constant_features.csv\',index=False)\n    corr=train.corr();corr.to_csv(out/\'feature_correlations_train.csv\')\n    pairs=[(a,b,corr.loc[a,b]) for i,a in enumerate(corr) for b in corr.columns[i+1:] if abs(corr.loc[a,b])>=.95]\n    pd.DataFrame(pairs,columns=[\'feature_a\',\'feature_b\',\'pearson_r\']).to_csv(out/\'high_correlation_pairs.csv\',index=False)\n    # Exclude all path geometry: direct target construction inputs/proxies.\n    # Also exclude ridge-to-path relations and position features that recreate centre.\n    forbidden=lambda c: c.startswith((\'path_\',\'normalized_path\',\'absolute_path\',\'missing_path\',\'ridge_path\',\'ridge_proximity\'))\n    broad=[c for c in train if not forbidden(c) and not c.endswith(\'_missing\') and train[c].nunique()>1 and train[c].isna().mean()<.2]\n    candidates=[\'cassava_imbalance\',\'ridge_imbalance\',\'ridge_orientation\',\'cassava_lower_obstruction\',\'edge_density\',\'glcm_contrast\',\'excess_green\',\'cassava_density\']\n    anfis=[]\n    for c in candidates:\n        if c in broad and all(not np.isfinite(corr.loc[c,d]) or abs(corr.loc[c,d])<.90 for d in anfis):anfis.append(c)\n        if len(anfis)==5:break\n    selected=anfis.copy()\n    for c in broad:\n        if c not in selected and all(not np.isfinite(corr.loc[c,d]) or abs(corr.loc[c,d])<.95 for d in selected):selected.append(c)\n    # ANFIS priorities are selected first using training-only redundancy checks.\n    (ROOT/\'configs/recommended_features.yaml\').write_text(yaml.safe_dump(dict(selection_split=\'train\',classical_ml=selected,anfis=anfis,excluded=\'All path-derived geometry; direct formula leakage. GT-mask features remain oracle-only until predicted masks are available.\')))\n    dictionary=[]\n    for c in numeric:\n        category=\'colour\' if c.startswith((\'rgb\',\'hsv\',\'excess\',\'normalized_green\')) else \'texture\' if c.startswith((\'glcm\',\'lbp\',\'texture\')) else \'edge\' if c.startswith(\'edge\') else \'navigation_geometry\' if c.startswith((\'cassava_\',\'ridge_path\',\'ridge_proximity\',\'normalized_path\',\'absolute_path\',\'missing_path\')) or c in [\'path_center_x\',\'path_width_lower\',\'path_continuity\',\'path_left_boundary\',\'path_right_boundary\'] or c.startswith(\'path_width_y\') else \'segmentation_geometry\'\n        unit=\'dimensionless; see formula\'\n        formula=\'Defined by extract() in src/features/feature_extraction.py: \'+c\n        if c.endswith(\'_missing\'): formula=f\'1 if {c[:-8]} is NaN/nonfinite, otherwise 0\';unit=\'0 or 1\'\n        elif \'_hist_\' in c: formula=\'bin count / number of pixels; equal-width bins\';unit=\'[0,1]\'\n        elif c.endswith(\'area_ratio\') or \'coverage\' in c or c==\'edge_density\':formula=\'foreground pixel count / region pixel count\';unit=\'[0,1]\'\n        elif c.endswith(\'centroid_x\'):formula=\'mean x coordinate of foreground pixel centres\';unit=\'pixels\'\n        elif c.endswith(\'centroid_y\'):formula=\'mean y coordinate of foreground pixel centres\';unit=\'pixels\'\n        elif c.endswith(\'polygon_count\'):formula=\'number of valid annotated instances\';unit=\'count\'\n        elif c.endswith(\'mask_area\'):formula=\'count of union foreground pixels\';unit=\'pixels squared\'\n        elif c.endswith(\'largest_polygon_area\'):formula=\'max absolute shoelace area across polygons\';unit=\'pixels squared\'\n        elif c.endswith(\'mean_polygon_area\'):formula=\'mean absolute shoelace area across polygons\';unit=\'pixels squared\'\n        elif c.endswith(\'perimeter\') and not c.startswith(\'edge\'):formula=\'sum of closed polygon Euclidean edge lengths\';unit=\'pixels\'\n        elif c.endswith(\'bounding_width\'):formula=\'max(x)-min(x)+1 over union mask\';unit=\'pixels\'\n        elif c.endswith(\'bounding_height\'):formula=\'max(y)-min(y)+1 over union mask\';unit=\'pixels\'\n        elif c.endswith(\'aspect_ratio\'):formula=\'bounding_width / bounding_height\'\n        elif c.endswith((\'leftmost\',\'rightmost\',\'topmost\',\'bottommost\')):formula=\'min x, max x, min y or max y of union mask as named\';unit=\'pixels\'\n        elif c.startswith(\'path_width\'):formula=\'outer right-left+1 divided by image width; lower ROI uses median row span; path_width is full-mask bounding width / image width\';unit=\'[0,1]\'\n        elif c==\'path_center_x\':formula=\'(median lower-ROI left boundary + median lower-ROI right boundary)/2\';unit=\'pixels\'\n        elif c in [\'path_left_boundary\',\'path_right_boundary\']:formula=\'median outer boundary coordinate across occupied lower-ROI rows\';unit=\'pixels\'\n        elif c==\'normalized_path_offset\':formula=\'(path_center_x - width/2)/(width/2)\';unit=\'[-1,1]\'\n        elif c==\'absolute_path_deviation\':formula=\'abs(normalized_path_offset)\';unit=\'[0,1]\'\n        elif c==\'path_continuity\':formula=\'fraction lower-ROI rows with >=1 path pixel\';unit=\'[0,1]\'\n        elif c==\'missing_path\':formula=\'1 if no path pixel else 0\';unit=\'0 or 1\'\n        elif c.endswith(\'imbalance\'):formula=\'left-half coverage minus right-half coverage\';unit=\'[-1,1]\'\n        elif c==\'cassava_lower_obstruction\':formula=\'leaf union pixel count in lower ROI / ROI area\';unit=\'[0,1]\'\n        elif c==\'cassava_density\':formula=\'leaf instance count / image megapixels\';unit=\'instances/Mpixel\'\n        elif c==\'ridge_orientation\':formula=\'principal eigenvector angle of ridge pixel covariance; NaN if eigenvalue ratio <=1.05\';unit=\'degrees [0,180)\'\n        elif c==\'ridge_proximity_to_path\':formula=\'mean ridge-pixel distance to nearest path pixel / image diagonal\';unit=\'[0,1]\'\n        elif c==\'ridge_path_overlap\':formula=\'ridge and path intersection area / ridge area\';unit=\'[0,1]\'\n        elif c==\'ridge_path_centroid_distance\':formula=\'Euclidean centroid distance / image diagonal\';unit=\'[0,1]\'\n        elif c.startswith((\'rgb\',\'hsv\')):formula=\'mean or population std of channel normalized to [0,1]; HSV hue is linear, not circular\';unit=\'[0,1]\'\n        elif c==\'excess_green\':formula=\'mean(2G-R-B), RGB in [0,1]\';unit=\'[-2,2]\'\n        elif c==\'excess_red\':formula=\'mean(1.4R-G), RGB in [0,1]\';unit=\'[-1,1.4]\'\n        elif c==\'normalized_green_red\':formula=\'mean((G-R)/(G+R+1e-8))\';unit=\'[-1,1]\'\n        elif c.startswith(\'glcm\'):formula=\'skimage graycoprops \'+c[5:]+\'; symmetric normalized 32-level GLCM, distance 1, mean over 0/45/90/135 degrees\'\n        elif c==\'texture_entropy\':formula=\'-sum(p*log2(p)) of grayscale histogram\';unit=\'bits [0,8]\'\n        elif c.startswith(\'lbp\'):formula=\'mean/std of uniform LBP(P=8,R=1), resized 256x256 grayscale\';unit=\'[0,9]\'\n        elif c==\'edge_contour_count\':formula=\'number of Canny contours (RETR_LIST)\';unit=\'count\'\n        elif c==\'edge_contour_mean_area\':formula=\'mean cv2.contourArea of Canny contours\';unit=\'pixels squared\'\n        elif c==\'edge_contour_mean_perimeter\':formula=\'mean closed Canny contour perimeter\';unit=\'pixels\'\n        elif c==\'edge_dominant_orientation\':formula=\'centre of maximum length-weighted Hough-line angle bin (18 bins)\';unit=\'degrees [0,180)\'\n        dictionary.append(dict(feature_name=c,category=category,meaning=c.replace(\'_\',\' \'),formula=formula,source=\'image\' if category in [\'colour\',\'texture\',\'edge\'] else \'ground-truth polygons (oracle); same implementation for predictions\',unit_range=unit,suitable_for_classical_ml=\'yes\' if c in selected else \'no\',suitable_for_ANFIS=\'yes\' if c in anfis else \'no\'))\n    pd.DataFrame(dictionary).to_csv(ROOT/\'data/processed/feature_dictionary.csv\',index=False)\n    counts=t.discrete_target.value_counts().reindex([\'left\',\'forward\',\'right\',\'stop_or_uncertain\'],fill_value=0)\n    counts.rename_axis(\'target\').to_csv(out/\'navigation_target_counts.csv\')\n    trainstats=f[f.split==\'train\'][[\'normalized_path_offset\',\'path_width_lower\',\'path_area_ratio\',\'path_continuity\']].describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]);trainstats.to_csv(out/\'target_threshold_training_distributions.csv\')\n    for col,label in [(\'normalized_offset\',\'Continuous offset\'),(\'path_width\',\'Path width / image width\'),(\'path_area_ratio\',\'Path area ratio\')]:\n        fig,ax=plt.subplots(figsize=(7,4));ax.hist(t[col].dropna(),bins=30,color=\'#287c72\');ax.set(xlabel=label,ylabel=\'Images\');save(fig,figdir/f\'{col}_distribution.png\')\n    fig,ax=plt.subplots(figsize=(7,4));counts.plot.bar(ax=ax,color=\'#287c72\',rot=15);ax.set(ylabel=\'Images\',xlabel=\'Provisional target\');save(fig,figdir/\'target_class_distribution.png\')\n    fig,ax=plt.subplots(figsize=(7,4));ax.scatter(t.path_width,t.normalized_offset,s=8,alpha=.4);ax.set(xlabel=\'Path width / image width\',ylabel=\'Normalized offset\');save(fig,figdir/\'offset_vs_width.png\')\n    chosen=t.groupby(\'discrete_target\',group_keys=False).sample(n=1,random_state=42).index.tolist()\n    chosen+=t.drop(index=chosen).sample(n=min(30,len(t))-len(chosen),random_state=42).index.tolist()\n    for n,i in enumerate(chosen):\n        row=t.iloc[i];m=mf.set_index(\'image_id\').loc[row.image_id];im=cv2.cvtColor(cv2.imread(str(ROOT/m.image_path)),cv2.COLOR_BGR2RGB);h,w=im.shape[:2];mask=masks_from_polygons(polygons(ROOT/m.label_path),h,w)[0]\n        fig,ax=plt.subplots(figsize=(7,7));ax.imshow(im);overlay=np.zeros((h,w,4));overlay[mask.astype(bool)]=[0,.9,.6,.35];ax.imshow(overlay);ax.axvline(w/2,color=\'white\',linestyle=\'--\',label=\'Image centre\')\n        fr=f.iloc[i]\n        for key,color,label in [(\'path_center_x\',\'yellow\',\'Path centre\'),(\'path_left_boundary\',\'cyan\',\'Left boundary\'),(\'path_right_boundary\',\'magenta\',\'Right boundary\')]:\n            if np.isfinite(fr[key]):ax.vlines(fr[key],roi_start*h,h,color=color,label=label)\n        ax.axhline(roi_start*h,color=\'white\',linestyle=\':\');ax.set_title(f\'{row.discrete_target} | offset={row.normalized_offset:.3f}\\n{row.image_id[:45]}\');ax.legend(loc=\'upper right\',fontsize=7);ax.axis(\'off\');save(fig,figdir/f\'diagnostic_{n+1:02d}.png\')\n    diagnostic=t.loc[chosen,[\'image_id\',\'split\',\'discrete_target\']].copy();diagnostic[\'figure\']=[f\'diagnostic_{n+1:02d}.png\' for n in range(len(chosen))];diagnostic.to_csv(out/\'navigation_diagnostic_manifest.csv\',index=False)\n    display=[\'path_width_lower\',\'path_area_ratio\',\'cassava_imbalance\',\'ridge_orientation\',\'edge_density\',\'normalized_path_offset\']\n    for group,labels in [(\'split\',f.split),(\'target\',t.discrete_target)]:\n        fig,axes=plt.subplots(2,3,figsize=(13,7))\n        for ax,c in zip(axes.flat,display):\n            groups=list(labels.unique());ax.boxplot([f.loc[labels==g,c].dropna() for g in groups],tick_labels=groups);ax.set_title(c);ax.tick_params(axis=\'x\',rotation=20)\n        save(fig,qdir/f\'distributions_by_{group}.png\')\n    f.assign(target=t.discrete_target).groupby(\'split\')[list(numeric)].agg([\'mean\',\'std\',\'median\']).to_csv(out/\'feature_distributions_by_split.csv\')\n    f.assign(target=t.discrete_target).groupby(\'target\')[list(numeric)].agg([\'mean\',\'std\',\'median\']).to_csv(out/\'feature_distributions_by_target.csv\')\n    fig,ax=plt.subplots(figsize=(8,7));im=ax.imshow(corr.loc[display,display],vmin=-1,vmax=1,cmap=\'coolwarm\');ax.set_xticks(range(6),display,rotation=60,ha=\'right\');ax.set_yticks(range(6),display);fig.colorbar(im,ax=ax);save(fig,qdir/\'correlations_train.png\')\n    hashes=mf.groupby(\'sha256\').split.nunique(); cross=int((hashes>1).sum())\n    pd.DataFrame([dict(check=\'exact_cross_split_hashes\',count=cross),dict(check=\'duplicate_image_ids\',count=mf.image_id.duplicated().sum()),dict(check=\'feature_target_id_mismatches\',count=(f.image_id!=t.image_id).sum())]).to_csv(out/\'leakage_checks.csv\',index=False)\n    assert cross==0 and not mf.image_id.duplicated().any() and f.image_id.equals(t.image_id)\n    assert f.groupby(\'split\').size().to_dict()==mf.groupby(\'split\').size().to_dict()\n    rate=numeric.isna().mean().mean(); rawrate=numeric[[c for c in numeric if not c.endswith(\'_missing\')]].isna().mean().mean()\n    (out/\'feature_quality_report.md\').write_text(f\'\'\'# Phase A feature quality\\n\\nProcessed {len(f)} images: {f.split.value_counts().to_dict()}. Historical nominal 603 train included one excluded checkpoint duplicate; no split assignments changed. Actual exclusions are in phaseA_pretraining_quality_check.csv.\\n\\n{len(numeric.columns)} features including explicit missing flags. Missing rate {rate:.4%}; excluding missing flags {rawrate:.4%}. Missing geometry remains NaN; valid zero coverage/counts remain zero. Train-derived constant/near-constant reports and correlations are separate CSVs. Near-constant means dominant value >=99%.\\n\\nClassical ML: {len(selected)} recommended features, listed in configs/recommended_features.yaml. Greedy correlation pruning uses train only at |r| >=0.95; no target outcomes or test tuning. ANFIS: {\', \'.join(anfis)}. Missing-value imputation/scaling must be fitted on train only in Phase B.\\n\\nAll path-derived features (including offset, centroid, boundaries, widths, area and continuity) are excluded from primary recommended inputs because targets are constructed from them. Using offset to predict offset is identity leakage; using width/continuity to predict stop reproduces the rule. Such geometry can only be used in explicitly labelled rule-reconstruction experiments. Ridge-to-path relations are excluded too.\\n\\nGT-mask features are oracle inputs unavailable to the robot. Predicted-mask features and honest out-of-fold train predictions are required for end-to-end evaluation. A segmentation model trained on all train images produces in-sample train masks, not out-of-fold features.\\n\\nExact cross-split image duplicates: {cross}. This does not rule out adjacent video frames or shared field sessions; grouping metadata is unavailable. Preserve existing splits but investigate temporal/site leakage before performance claims. Feature distributions by split/target are descriptive only.\\n\\nRaster masks use independent union fills at native resolution. Polygon areas/perimeters are continuous shoelace/edge calculations; very small polygons may rasterize to a pixel and are retained. Row widths are outer spans and can bridge disjoint islands. Continuity measures row occupancy, not physical connectivity or traversability. Hue statistics are linear. Texture uses 256-pixel resampling.\\n\'\'\')\n    (out/\'navigation_target_report.md\').write_text(f\'\'\'# Provisional navigation targets\\n\\nLabels are geometry-derived, not manually recorded robot controls. Continuous offset is the primary research target: (path_center_x - width/2)/(width/2). Negative means image-left, positive image-right; this is not a calibrated wheel/steering sign. Centre is the midpoint of median outer boundaries over occupied rows in the lower {100*(1-roi_start):.0f}% of the image.\\n\\n{counts.to_string()}\\n\\nContinuous target is available for {t.continuous_target.notna().sum()} of {len(t)} images. Missing centres remain NaN. Low-quality finite offsets remain available but warning flags must be used to define a training/evaluation policy.\\n\\nThresholds in configs/navigation_targets.yaml are provisional. Deadband and minimum area/width/continuity are derived from training-only quantiles with declared bounds, or explicit user overrides. Exact values and derivation are in configs/navigation_targets.yaml and target_threshold_training_distributions.csv. No test outcomes tune thresholds. Confidence is min(clipped area/min-area, clipped width/min-width, continuity); it is a heuristic, not a calibrated probability. Stop/uncertain combines threshold failures, missing centre, and confidence below the configured stop_uncertain_threshold. Smoothing disabled: no validated temporal order.\\n\\nReproducible diagnostic overlays (30 for datasets of at least 30 images) include image centre, path union, lower ROI, available path centre/boundaries, offset and class. Their IDs are in navigation_diagnostic_manifest.csv. Missing path geometry cannot be drawn.\\n\\nThe control team must later validate these targets against the intended physical controller. Phase B models are research navigation-prediction models, not direct motor-control systems. Perspective, camera calibration, obstacle depth, multiple disjoint paths, mask errors and temporal/site leakage remain unresolved. No physical robot safety or controller accuracy is claimed.\\n\'\'\')\n    # Contact sheet complements the 30 full-resolution individual diagnostics.\n    fig,axes=plt.subplots(2,3,figsize=(15,10))\n    for ax in axes.flat:ax.axis(\'off\')\n    for n,ax in enumerate(axes.flat):\n        path=figdir/f\'diagnostic_{n+1:02d}.png\'\n        if path.exists():ax.imshow(plt.imread(path))\n    fig.suptitle(\'Provisional geometry-derived navigation targets\',fontsize=16)\n    save(fig,figdir/\'navigation_target_examples.png\')\n    # Explicit train-only feature-target relationships are descriptive, not test selection.\n    rel=train.corrwith(t.loc[f.split==\'train\',\'continuous_target\']).rename(\'train_pearson_r_to_offset\')\n    rel.rename_axis(\'feature\').to_csv(out/\'feature_target_relationships_train.csv\')\n    stats=dict(images=len(f),features=len(numeric.columns),missing_rate=rate,ml_count=len(selected),anfis=anfis,targets=counts.to_dict())\n    print(stats)\n    return stats\n\nif __name__==\'__main__\':run()\n', encoding='utf-8')
print('Created', helper_path.relative_to(PROJECT))


## 7. Train one segmentation baseline

Defaults: 640 pixels, 60 epochs, batch 8, patience 12, seed 42, two workers, AdamW. Augmentations are mild rotation/translation/scaling and HSV brightness/saturation; vertical flipping, mosaic and mixup are disabled. Horizontal mirroring is disabled unless explicitly enabled above. Implicit version/environment-dependent Albumentations are disabled.

Validation alone selects the best checkpoint through the installed Ultralytics fitness function. The exact best epoch is captured when the checkpoint is saved. Test is evaluated only after selection. Detailed output goes to `training.log`; a progress message appears every five epochs. Rerunning a completed training cell reuses weights only if configuration, images and labels match. An incomplete run stops with guidance instead of silently restarting or mixing outputs. Colab may disconnect: download/save checkpoints before resetting a runtime.

In [ ]:
from phase_a_segmentation import train_one, evaluate, predict_all
TRAIN_CONFIG=dict(imgsz=IMAGE_SIZE,epochs=EPOCHS,batch=BATCH_SIZE,patience=PATIENCE,workers=WORKERS,
                  seed=SEED,deterministic=True,optimizer='AdamW',lr0=LEARNING_RATE,lrf=0.01,
                  hsv_h=0.01,hsv_s=0.20,hsv_v=0.20,degrees=5.0,translate=0.05,scale=0.15,
                  flipud=0.0,fliplr=HORIZONTAL_FLIP,mosaic=0.0,mixup=0.0,copy_paste=0.0,
                  shear=0.0,perspective=0.0)
MODEL,TRAIN_STATE=train_one(PROJECT,TRAIN_CONFIG,MANIFEST,RUN_NAME)
print('Checkpoints:',PROJECT/'models/segmentation_baseline/best.pt',PROJECT/'models/segmentation_baseline/last.pt')


## 8. Evaluate the frozen model and generate predictions

Metrics are read from actual Ultralytics objects and raw result dictionaries; unavailable values stay missing. Validation and test metrics are saved separately. PR/F1/confusion figures are preserved when the installed API emits them, with an availability CSV. All test predictions are saved (at least 20).

Timing uses three warm-ups, then batch-one inference. `inference_ms` includes model-reported preprocessing + inference + postprocessing; `wall_ms` includes the whole predict call and image read. FPS is 1000 / mean pipeline latency on the recorded GPU. It is not measured robot throughput. Semantic union IoU ranks good/difficult test examples for inspection only; it is not instance mask AP and does not tune the model.

In [ ]:
METRICS,PER_CLASS=evaluate(PROJECT,MODEL,TRAIN_STATE,TRAIN_CONFIG)
PREDICTED_POLYGONS,INFERENCE=predict_all(PROJECT,MODEL,MANIFEST,IMAGE_SIZE,PREDICTION_CONFIDENCE)
METRICS=pd.read_csv(PROJECT/'results/segmentation_baseline/metrics.csv')
display(METRICS.round(4))
print('Saved test_prediction_examples.png, failure_cases.png and all test predictions.')


## 9. Extract identical features from ground-truth and predicted polygons

Native-resolution union masks avoid double-counting overlapping instances. Continuous polygon areas/perimeters use their coordinates. Empty masks have valid zero counts/coverage; undefined centroids, widths and orientations remain NaN with explicit missing flags. Image IDs, original paths, split, extraction status and warnings are retained.

Colour, texture (256×256 GLCM/LBP) and edge descriptors come from the same original image in both tables. Predicted geometry uses the model's normalized instance polygons and the identical rasterization/features as ground truth. Lower-ROI width is the median outer row span; disjoint regions can therefore overestimate traversable width. Row continuity measures occupancy, not connectivity.

Ground-truth masks are **oracle inputs**. Predicted training masks are **in-sample**, because segmentation saw those images. Later honest end-to-end modeling needs out-of-fold predicted training masks; these tables do not supply them.

In [ ]:
FEATURE_CONFIG=dict(seed=SEED,dataset_root=str(DATASET_ROOT),lower_roi_start=LOWER_ROI_START,
                    width_rows=[0.5,0.65,0.8,0.95],texture_size=256,glcm_levels=32,
                    histogram_bins=16,canny_thresholds=[100,200])
(PROJECT/'configs/features.yaml').write_text(yaml.safe_dump(FEATURE_CONFIG,sort_keys=False))
from features.feature_extraction import run as extract_features
FEATURES_GT=extract_features()
FEATURES_PRED=extract_features(PREDICTED_POLYGONS)
assert FEATURES_GT.image_id.equals(FEATURES_PRED.image_id)
print('Ground-truth feature rows:',len(FEATURES_GT),'| Predicted-mask rows:',len(FEATURES_PRED))


## 10. Fit provisional target thresholds using training distributions only

Negative normalized offset means the path centre lies left of the image centre; positive means right. The primary target is `(path_center_x - image_width/2)/(image_width/2)`.

The symmetric left/right deadband uses a configurable quantile of training absolute offsets. Minimum area, width and continuity use training lower quantiles. Declared bounds avoid pathological values; they are research assumptions, **not controller settings**. Overrides can be set near the top. Empty training geometry stops target fitting. The fitted YAML records every value and its derivation.

`stop_or_uncertain` means missing/weak path geometry or an annotation technical-row exclusion. Confidence is a geometric quality heuristic, not a probability. Finite continuous offsets are retained even for uncertain rows, with warnings; Phase B must explicitly choose a quality filter. Temporal smoothing is disabled because verified sequence IDs/order are unavailable.

In [ ]:
train_geometry=FEATURES_GT.loc[FEATURES_GT.split=='train',
    ['normalized_path_offset','path_width_lower','path_area_ratio','path_continuity']]
train_geometry.describe(percentiles=[.05,.1,.25,.35,.5,.75,.9,.95]).to_csv(PROJECT/'reports/target_threshold_training_distributions.csv')
def bounded_quantile(series: pd.Series, quantile: float, bounds: list) -> float:
    """Fit one provisional threshold on finite positive training measurements."""
    values=series.replace([np.inf,-np.inf],np.nan).dropna()
    values=values[values>0]
    if values.empty: raise ValueError('Insufficient training path geometry to derive thresholds.')
    return float(np.clip(values.quantile(quantile),*bounds))
q=THRESHOLD_POLICY['minimum_geometry_quantile']
# Absolute zero offsets are valid for deadband fitting, so retain them here.
abs_offsets=train_geometry.normalized_path_offset.abs().dropna()
assert len(abs_offsets)>0, 'No finite training path centres.'
deadband=float(np.clip(abs_offsets.quantile(THRESHOLD_POLICY['deadband_abs_offset_quantile']),*THRESHOLD_POLICY['deadband_bounds']))
TARGET_CONFIG=dict(status='provisional',seed=SEED,lower_roi_start=LOWER_ROI_START,
    left_threshold=-deadband,right_threshold=deadband,
    minimum_path_area_ratio=bounded_quantile(train_geometry.path_area_ratio,q,THRESHOLD_POLICY['area_bounds']),
    minimum_valid_path_width=bounded_quantile(train_geometry.path_width_lower,q,THRESHOLD_POLICY['width_bounds']),
    minimum_path_continuity=bounded_quantile(train_geometry.path_continuity,q,THRESHOLD_POLICY['continuity_bounds']),
    smoothing={'enabled':False,'reason':'No validated sequence identifiers'},
    derivation={'split':'train','policy':THRESHOLD_POLICY,'overrides':THRESHOLD_OVERRIDES},
    stop_uncertain_rules=['missing centre','area below minimum','width below minimum',
                          'row continuity below minimum','confidence below threshold','technical annotation rows excluded'])
TARGET_CONFIG['stop_uncertain_threshold']=TARGET_CONFIG['minimum_path_continuity']
allowed={'left_threshold','right_threshold','minimum_path_area_ratio','minimum_valid_path_width','minimum_path_continuity','stop_uncertain_threshold'}
assert set(THRESHOLD_OVERRIDES)<=allowed, 'Unknown threshold override.'
TARGET_CONFIG.update(THRESHOLD_OVERRIDES)
assert -1 <= TARGET_CONFIG['left_threshold'] < TARGET_CONFIG['right_threshold'] <= 1
assert all(0<TARGET_CONFIG[k]<=1 for k in ['minimum_path_area_ratio','minimum_valid_path_width','minimum_path_continuity','stop_uncertain_threshold'])
(PROJECT/'configs/navigation_targets.yaml').write_text(yaml.safe_dump(TARGET_CONFIG,sort_keys=False))
from navigation.target_generation import run as generate_targets
TARGETS=generate_targets()
print('Fitted provisional thresholds:',{k:TARGET_CONFIG[k] for k in sorted(allowed)})
print(TARGETS.discrete_target.value_counts().to_string())


## 11. Target diagnostics, feature quality and recommendations

Generate at least 30 diagnostic overlays, distributions, train-only correlations, missingness, constant/near-constant features and a feature dictionary. The ANFIS recommendation uses 3–6 interpretable nonredundant variables and training statistics only.

**Leakage warning:** path centre/offset directly define the continuous target. Path width/area/continuity also define the discrete uncertainty rule. These are available in the feature table for interpretation and explicit rule-reconstruction experiments, but excluded from the primary prediction feature recommendations. Selecting them for a model that predicts their own formula would create misleading accuracy. This takes precedence over prioritizing path geometry for ANFIS. The recommended features instead use cassava imbalance, ridge geometry and image cues.

In [ ]:
from phase_a_analysis import run as analyze_phase_a
ANALYSIS=analyze_phase_a()
TARGETS=pd.read_csv(PROJECT/'data/processed/navigation_targets.csv')
RECOMMENDATIONS=yaml.safe_load((PROJECT/'configs/recommended_features.yaml').read_text())
print('ML features:',len(RECOMMENDATIONS['classical_ml']))
print('ANFIS:',RECOMMENDATIONS['anfis'])
from IPython.display import Image,display
display(Image(filename=str(PROJECT/'figures/navigation_targets/navigation_target_examples.png'),width=1000))
display(Image(filename=str(PROJECT/'figures/segmentation_baseline/test_prediction_examples.png'),width=1000))


## 12. Reports, verification and output inventory

In [ ]:
import hashlib
segmentation_report = '# Phase A segmentation baseline\n\n'
segmentation_report += METRICS.to_markdown(index=False) + '\n\n'
segmentation_report += 'Actual environment: ' + json.dumps(ENVIRONMENT) + '\n\n'
segmentation_report += ('One architecture trained. Best epoch captured at checkpoint save; validation-only selection. '
    'Test evaluated after freezing the checkpoint. Best and last weights preserved. '
    'Training settings are in configs/yolo_baseline.yaml; detailed logs and raw API metrics are retained. '
    'Unavailable API fields stay missing (N/A), not zero. Figure availability is recorded separately.\n\n'
    'Batch-one timing follows three warm-ups: inference_ms sums preprocess, model inference and postprocess; '
    'wall_ms also includes the full predict call/image read. FPS is reciprocal mean pipeline latency on this GPU. '
    'Semantic union IoU ranks representative/difficult test examples only; it is not instance AP. '
    'All usable test images have prediction examples. Any entirely undetected class cannot have a representative prediction.\n\n'
    'GT feature tables use oracle masks. Predicted training masks are in-sample; out-of-fold training masks are required '
    'for honest downstream evaluation. Exact duplicate checks do not rule out temporal/site leakage. '
    'No navigation model or robot controller was trained.\n\n'
    'Sources: [Ultralytics segmentation](https://docs.ultralytics.com/tasks/segment/), '
    '[training](https://docs.ultralytics.com/modes/train/).\n')
(PROJECT/'reports/segmentation_baseline_report.md').write_text(segmentation_report)
# Preserve exact fitted thresholds in the navigation report, beyond the narrative explanation.
nav_report=PROJECT/'reports/navigation_target_report.md'
nav_report.write_text(nav_report.read_text()+'\n## Fitted provisional configuration\n\n```yaml\n'+yaml.safe_dump(TARGET_CONFIG,sort_keys=False)+'```\n')
for table in [FEATURES_GT,FEATURES_PRED,TARGETS]:
    assert not table.image_id.duplicated().any()
    assert table.image_id.tolist()==MANIFEST.image_id.tolist()
    assert table.split.tolist()==MANIFEST.split.tolist()
assert (MANIFEST.groupby('sha256').split.nunique()<=1).all()
assert set(TARGETS.discrete_target)<= {'left','forward','right','stop_or_uncertain'}
assert yaml.safe_load((PROJECT/'configs/classes.yaml').read_text())=={'names':{0:'path',1:'cassava_leaves',2:'ridge'},'status':'confirmed'}
assert (PROJECT/'models/segmentation_baseline/best.pt').stat().st_size>0
assert (PROJECT/'models/segmentation_baseline/last.pt').stat().st_size>0
assert len(list((PROJECT/'figures/navigation_targets').glob('diagnostic_*.png')))>=30
assert len(list((PROJECT/'figures/segmentation_baseline/test_predictions').glob('*.png')))>=20
if len(RECOMMENDATIONS['anfis'])<3:
    print('Insufficient variable, nonredundant ANFIS features; Phase B readiness will be No.')
required=[
 'results/segmentation_baseline/metrics.csv','results/segmentation_baseline/per_class_metrics.csv',
 'results/segmentation_baseline/inference_results.csv','results/segmentation_baseline/failure_cases.csv',
 'data/processed/features_ground_truth.csv','data/processed/features_predicted_masks.csv',
 'data/processed/navigation_targets.csv','data/processed/feature_dictionary.csv',
 'reports/phaseA_pretraining_quality_check.csv','reports/feature_extraction_summary.csv',
 'reports/feature_missing_values.csv','reports/high_correlation_pairs.csv',
 'reports/segmentation_baseline_report.md','reports/navigation_target_report.md','reports/feature_quality_report.md',
 'figures/segmentation_baseline/test_prediction_examples.png','figures/segmentation_baseline/failure_cases.png',
 'figures/navigation_targets/navigation_target_examples.png',
 'configs/classes.yaml','configs/cassava_segmentation.yaml','configs/yolo_baseline.yaml',
 'configs/features.yaml','configs/navigation_targets.yaml']
assert all((PROJECT/p).is_file() for p in required)
checks={'feature_target_alignment':True,'unique_image_ids':True,'exact_cross_split_hash_leakage':False,
        'seed':SEED,'test_used_for_tuning':False,'phase_b_trained':False,
        'ground_truth_rows':len(FEATURES_GT),'predicted_rows':len(FEATURES_PRED),
        'limits':'Exact hashes cannot establish independence of video frames/sites; no OOF training predictions.'}
(PROJECT/'reports/phaseA_verification.json').write_text(json.dumps(checks,indent=2))
print('Verified alignment, class mapping, checkpoints, required CSVs/reports and diagnostic counts.')
print('Ready for Phase B research preparation; honest end-to-end evaluation still needs OOF train masks and control-team validation.')


## 13. Build the downloadable results ZIP

Includes actual CSVs, reports, figures, reusable source, configuration, raw result files, and **best + last checkpoints**. The input archive and dataset images/labels are excluded. Copies of intermediate checkpoints inside the training run are omitted to avoid duplicate large weights.

The executed notebook itself is downloaded separately with **File > Download > Download .ipynb**. Return **both** that executed notebook and `cassava_navigation_phase_A_results.zip` for review. Download before the runtime disconnects.

In [ ]:
import zipfile
RESULTS_ZIP=Path('/content/cassava_navigation_phase_A_results.zip')
archive_files=[]
for directory in ['configs','reports','figures','src','models','results']:
    for path in sorted((PROJECT/directory).rglob('*')):
        if not path.is_file() or '__pycache__' in path.parts or path.name=='output_inventory.csv':continue
        relative=path.relative_to(PROJECT)
        if path.suffix in {'.pt','.pth'} and not str(relative).startswith('models/segmentation_baseline/'):
            continue
        archive_files.append(path)
archive_files += sorted((PROJECT/'data/processed').glob('*.csv'))
assert archive.resolve() not in [p.resolve() for p in archive_files]
inventory=[{'path':str(p.relative_to(PROJECT)),'bytes':p.stat().st_size,
            'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in archive_files]
pd.DataFrame(inventory).to_csv(PROJECT/'reports/output_inventory.csv',index=False)
archive_files.append(PROJECT/'reports/output_inventory.csv')
with zipfile.ZipFile(RESULTS_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for path in sorted(set(archive_files)):
        z.write(path,arcname=str(path.relative_to(PROJECT)))
with zipfile.ZipFile(RESULTS_ZIP) as z:
    assert z.testzip() is None
    assert 'models/segmentation_baseline/best.pt' in z.namelist()
    assert 'models/segmentation_baseline/last.pt' in z.namelist()
print('Results ZIP:',RESULTS_ZIP,'| Size MB:',round(RESULTS_ZIP.stat().st_size/1e6,2))


## 14. Download and actual final summary

All values below come from this run's files. `N/A` means unavailable. “Ready” is limited to research modeling preparation; it does not mean a validated motor controller. Download the executed notebook separately from the File menu.

In [ ]:
from google.colab import files
if DOWNLOAD_RESULTS:
    files.download('/content/cassava_navigation_phase_A_results.zip')

def fmt(value: object) -> str:
    """Format available scalars without inventing missing results."""
    if value is None:return 'N/A'
    if isinstance(value,(int,float,np.number)):
        return f'{value:.5g}' if np.isfinite(value) else 'N/A'
    return str(value)
METRICS=pd.read_csv(PROJECT/'results/segmentation_baseline/metrics.csv')
v=METRICS.set_index('split').loc['valid'];t=METRICS.set_index('split').loc['test']
counts=TARGETS.discrete_target.value_counts()
numeric=FEATURES_GT.select_dtypes(include='number').drop(columns=['image_width','image_height'])
ready=bool(len(RECOMMENDATIONS['anfis'])>=3 and RESULTS_ZIP.exists())
summary=f"""PHASE A COMPLETE

DATASET
Train images: {int((MANIFEST.split=='train').sum())}
Validation images: {int((MANIFEST.split=='valid').sum())}
Test images: {int((MANIFEST.split=='test').sum())}
Classes:
0 = path
1 = cassava_leaves
2 = ridge

SEGMENTATION
Model: {TRAIN_STATE['architecture']}
Best epoch: {fmt(TRAIN_STATE['best_epoch'])}
Validation mask mAP@0.5: {fmt(v.get('mask_map50'))}
Validation mask mAP@0.5:0.95: {fmt(v.get('mask_map50_95'))}
Test mask mAP@0.5: {fmt(t.get('mask_map50'))}
Test mask mAP@0.5:0.95: {fmt(t.get('mask_map50_95'))}
Precision: {fmt(v.get('mask_precision'))} (validation mask)
Recall: {fmt(v.get('mask_recall'))} (validation mask)
Inference time: {fmt(t.get('inference_ms'))} ms/image (test pipeline)
FPS: {fmt(t.get('fps'))}
Model size: {fmt(t.get('model_size_mb'))} MB

FEATURE EXTRACTION
Images processed: {len(MANIFEST)}
Ground-truth feature rows: {len(FEATURES_GT)}
Predicted-mask feature rows: {len(FEATURES_PRED)}
Number of extracted features: {len(numeric.columns)} (including missing flags)
Missing-value rate: {numeric.isna().mean().mean():.4%}
Recommended ML feature count: {len(RECOMMENDATIONS['classical_ml'])}
Recommended ANFIS features: {', '.join(RECOMMENDATIONS['anfis'])}

NAVIGATION TARGETS
Continuous targets generated: {TARGETS.continuous_target.notna().sum()} / {len(TARGETS)}
Discrete targets generated: {len(TARGETS)}
Left: {counts.get('left',0)}
Forward: {counts.get('forward',0)}
Right: {counts.get('right',0)}
Stop/uncertain: {counts.get('stop_or_uncertain',0)}
Threshold status: provisional

OUTPUTS
Results ZIP: {RESULTS_ZIP}
Best checkpoint: {PROJECT/'models/segmentation_baseline/best.pt'}
Reports: {PROJECT/'reports'}
Figures: {PROJECT/'figures'}

READY FOR PHASE B:
{'Yes' if ready else 'No'} (research preparation only)

LIMITATIONS:
Geometry-derived labels are not recorded controls. Control-team validation is required.
GT masks are oracle inputs; predicted train masks are in-sample, not out-of-fold.
Avoid target-formula leakage. Temporal/site leakage remains possible beyond exact hashes.
Row-span path width can bridge disconnected masks; no depth/camera/controller calibration.
Unavailable API metrics/figures remain N/A. No direct motor-control or safety claim.
"""
print(summary)
